In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS gold;

In [0]:
%sql
CREATE OR REPLACE TABLE aml_risk_prediction.gold.dim_account
USING DELTA
AS
SELECT DISTINCT
    xxhash64(bank_id, account_number) AS account_key,
    account_number,
    bank_id,
    bank_name,
    entity_id,
    entity_name
FROM aml_risk_prediction.silver.silver_table2;

In [0]:
%sql
SELECT * FROM aml_risk_prediction.gold.dim_account

In [0]:
%sql
CREATE OR REPLACE TABLE aml_risk_prediction.gold.dim_bank
USING DELTA
AS
SELECT DISTINCT
    xxhash64(bank_id) AS bank_key,
    bank_id,
    bank_name
FROM aml_risk_prediction.silver.silver_table2;

In [0]:
%sql
SELECT * FROM aml_risk_prediction.gold.dim_bank

In [0]:
%sql
CREATE OR REPLACE TABLE aml_risk_prediction.gold.dim_date
USING DELTA
AS
SELECT DISTINCT
    CAST(date_format(CAST(timestamp AS DATE), 'yyyyMMdd') AS INT) AS date_key,
    CAST(timestamp AS DATE) AS transaction_date,
    YEAR(timestamp) AS year,
    QUARTER(timestamp) AS quarter,
    MONTH(timestamp) AS month,
    date_format(timestamp, 'MMMM') AS month_name,
    DAY(timestamp) AS day
FROM aml_risk_prediction.silver.silver_transactions
WHERE timestamp IS NOT NULL;

In [0]:
%sql
SELECT * FROM aml_risk_prediction.gold.dim_date LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE TABLE aml_risk_prediction.gold.dim_payment_format
USING DELTA
AS
SELECT DISTINCT
    xxhash64(payment_format) AS payment_format_key,
    payment_format
FROM aml_risk_prediction.silver.silver_transactions
WHERE payment_format IS NOT NULL;

In [0]:
%sql
CREATE OR REPLACE TABLE aml_risk_prediction.gold.dim_currency
USING DELTA
AS
SELECT DISTINCT
    xxhash64(currency) AS currency_key,
    currency
FROM (
    SELECT payment_currency AS currency
    FROM aml_risk_prediction.silver.silver_transactions

    UNION

    SELECT receiving_currency AS currency
    FROM aml_risk_prediction.silver.silver_transactions
)
WHERE currency IS NOT NULL;

In [0]:
%sql
SELECT *
FROM aml_risk_prediction.gold.fact_transaction
LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW account_behavior AS

SELECT
    account_key,

    COUNT(*) AS account_transaction_count,

    ROUND(AVG(amount_paid),2) AS account_avg_amount,

    ROUND(SUM(amount_paid),2) AS account_total_amount

FROM aml_risk_prediction.gold.fact_transaction

GROUP BY account_key;

In [0]:
%sql
SELECT * FROM account_behavior LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE TABLE aml_ml_features AS

SELECT

    -- Transaction identification
    t.transaction_key,

    -- Account
    t.account_key,

    -- Transaction amounts
    t.amount_paid,
    t.amount_received,

    -- Account behavior
    ab.account_transaction_count,
    ab.account_avg_amount,
    ab.account_total_amount,

    -- Payment information
    pf.payment_format,

    -- Currency information
    pc.currency AS payment_currency,
    rc.currency AS receiving_currency,

    -- Date information
    d.transaction_date,
    d.year AS transaction_year,
    d.quarter AS transaction_quarter,

    -- Currency changed?
    CASE
        WHEN t.payment_currency_key != t.receiving_currency_key
        THEN 1
        ELSE 0
    END AS currency_changed,

    -- Is transaction unusually large?
    CASE
        WHEN ab.account_avg_amount > 0
             AND t.amount_paid / ab.account_avg_amount >= 5
        THEN 1
        ELSE 0
    END AS is_large_transaction,

    -- How large is it compared to normal account behavior?
    CASE
        WHEN ab.account_avg_amount > 0
        THEN t.amount_paid / ab.account_avg_amount
        ELSE 0
    END AS amount_vs_account_avg,

    -- TARGET
    t.is_laundering

FROM aml_risk_prediction.gold.fact_transaction t

LEFT JOIN account_behavior ab
    ON t.account_key = ab.account_key

LEFT JOIN aml_risk_prediction.gold.dim_payment_format pf
    ON t.payment_format_key = pf.payment_format_key

LEFT JOIN aml_risk_prediction.gold.dim_currency pc
    ON t.payment_currency_key = pc.currency_key

LEFT JOIN aml_risk_prediction.gold.dim_currency rc
    ON t.receiving_currency_key = rc.currency_key

LEFT JOIN aml_risk_prediction.gold.dim_date d
    ON t.date_key = d.date_key;

In [0]:
%sql
SELECT * FROM aml_risk_prediction.gold.aml_ml_features 
LIMIT 10   ;

In [0]:
%sql
DESCRIBE TABLE aml_risk_prediction.gold.aml_ml_features;

In [0]:
%sql
CREATE OR REPLACE TABLE aml_risk_prediction.gold.aml_ml_features AS
SELECT 
    account_key,
    transaction_count,
    unique_transactions,
    active_days,
    CAST(total_amount_paid AS DECIMAL(18, 2)) AS total_amount_paid,
    CAST(total_amount_received AS DECIMAL(18, 2)) AS total_amount_received,
    CAST(avg_amount_paid AS DECIMAL(18, 2)) AS avg_amount_paid,
    CAST(avg_amount_received AS DECIMAL(18, 2)) AS avg_amount_received,
    CAST(max_amount_paid AS DECIMAL(18, 2)) AS max_amount_paid,
    CAST(min_amount_paid AS DECIMAL(18, 2)) AS min_amount_paid,
    CAST(std_amount_paid AS DECIMAL(18, 2)) AS std_amount_paid,
    unique_from_banks,
    unique_to_banks,
    unique_payment_formats,
    unique_payment_currencies,
    unique_receiving_currencies,
    outlier_transaction_count,
    outlier_transaction_amount,
    laundering_flag,
    avg_transaction_value,
    outlier_transaction_ratio,
    transactions_per_active_day,
    bank_network_diversity,
    currency_diversity,
    payment_method_diversity
FROM aml_risk_prediction.gold.aml_ml_features;

In [0]:
%sql
SELECT * FROM aml_risk_prediction.gold.aml_ml_features
LIMIT 10;